In [0]:
%run ./Includes/Copy-Datasets

In [0]:
(spark.readStream.table("books").createOrReplaceTempView("books_stream_tmpview"))

In [0]:
%sql
select * from books_stream_tmpview;


In [0]:
%sql
select author,count(book_id) as count from books_stream_tmpview group by author;

In [0]:
%sql
select author,book_id as count from books_stream_tmpview order by author;

In [0]:
%sql
create or replace temp view author_cout_view as
( select 
author, count(book_id) as total_books 
from books_stream_tmpview group by author);     

In [0]:
spark.table("author_cout_view").writeStream.trigger(processingTime="4 seconds").outputMode("complete").option("checkpointLocation", "/tmp/books_checkpoint").table("author_counts")


In [0]:
%sql
select * from author_counts;

In [0]:
%sql
insert into books values 
  ("B19", "The Lord of the Rings","test3", "vhaubey",25),
  ("B20", "The Hobbit","test3", "vhaubey",30),
  ("B21", "The Silmarillion","test3", "vhaubey",35);

In [0]:
(spark.table("author_cout_view").writeStream.trigger(availableNow=True).outputMode("complete").option("checkpointLocation", "/tmp/books_checkpoint").table("author_counts").awaitTermination())


In [0]:
%sql
select * from author_counts;